# NHL Fight Momentum — Analysis

This notebook takes the cleaned fight dataset produced by `fight_data_extraction.ipynb` and works through three layers of evaluation:

1. **Exploratory Data Analysis** — Understand the shape of the data and surface early patterns
2. **Statistical Testing** — Use a paired t-test to determine whether the momentum shift after a fight is real or just noise
3. **Predictive Modeling** — Build two models to understand *when* and *for whom* a fight generates momentum

The central question throughout is: **does a team's offensive output measurably improve in the 5 minutes after a fight compared to the 5 minutes before?**

In [ ]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
from scipy import stats
from sklearn.ensemble import RandomForestRegressor, RandomForestClassifier
from sklearn.model_selection import train_test_split
from sklearn.metrics import (
    mean_absolute_error, r2_score,
    confusion_matrix, precision_recall_curve
)
import warnings
warnings.filterwarnings('ignore')

# Load the extracted fight dataset
df = pd.read_csv('NHL_Fight_Momentum_Data.csv')

# Add a three-class momentum label
# Rather than just yes/no, this captures whether a team surged, stayed neutral, or declined
# Thresholds are based on the natural distribution of Corsi_Delta in the dataset
def momentum_class(delta):
    if delta >= 2:
        return 'Surge'
    elif delta <= -2:
        return 'Decline'
    else:
        return 'Neutral'

df['Momentum_Class'] = df['Corsi_Delta'].apply(momentum_class)

print(f'Total fight observations loaded: {len(df):,}')
print(f'Seasons: {df["Season"].min()} to {df["Season"].max()}')
print()
print(df[['Corsi_Delta', 'xG_Delta', 'Gained_Momentum', 'Is_Home', 'Was_Trailing']].describe())

---
## Part 1 — Exploratory Data Analysis

Before running any models, we want to understand the data visually. EDA helps us spot patterns, check assumptions, and make sure nothing looks broken before we commit to a statistical approach.

We'll look at four things:
- How fights have trended over time
- The overall distribution of momentum shifts
- Whether trailing teams benefit more than leading teams
- Whether home teams benefit more than away teams

In [ ]:
# --- VISUAL 1: Fights Per Season ---
# This establishes context. Fighting has declined sharply since 2010, which means
# our later seasons have smaller sample sizes — worth keeping in mind.

fights_per_season = df.groupby('Season').size() // 2  # divide by 2 because each fight has 2 rows

plt.figure(figsize=(10, 5))
plt.plot(fights_per_season.index, fights_per_season.values,
         marker='o', linewidth=2.5, color='steelblue')
plt.fill_between(fights_per_season.index, fights_per_season.values, alpha=0.15, color='steelblue')
plt.title('Confirmed NHL Fights Per Season (2010–2023)', fontsize=14)
plt.xlabel('Season', fontsize=12)
plt.ylabel('Number of Fights', fontsize=12)
plt.grid(True, linestyle='--', alpha=0.5)
plt.tight_layout()
plt.savefig('outputs/Fights_Per_Season.png')
plt.show()

In [ ]:
# --- VISUAL 2: Distribution of Corsi Delta ---
# This shows the spread of momentum shifts across all fights.
# If fighting truly generates momentum, we would expect this distribution
# to be skewed positive. If it is centered at zero, fighting has no consistent effect.

plt.figure(figsize=(10, 5))
sns.histplot(df['Corsi_Delta'], bins=20, kde=True, color='steelblue')
plt.axvline(0, color='red', linestyle='--', label='No Change')
plt.axvline(df['Corsi_Delta'].mean(), color='green', linestyle='--',
            label=f'Mean ({df["Corsi_Delta"].mean():.2f})')
plt.title('Distribution of Corsi Delta After a Fight', fontsize=14)
plt.xlabel('Corsi Delta (After minus Before)', fontsize=12)
plt.ylabel('Count', fontsize=12)
plt.legend()
plt.grid(True, linestyle='--', alpha=0.4)
plt.tight_layout()
plt.savefig('outputs/Corsi_Delta_Distribution.png')
plt.show()

print(f'Mean Corsi Delta: {df["Corsi_Delta"].mean():.4f}')
print(f'Median Corsi Delta: {df["Corsi_Delta"].median():.4f}')
print(f'Teams that gained momentum: {(df["Corsi_Delta"] > 0).sum()} of {len(df)} ({(df["Corsi_Delta"] > 0).mean():.1%})')

In [ ]:
# --- VISUAL 3: Three-Class Momentum Breakdown ---
# The binary Gained_Momentum label misses nuance.
# Here we split into Surge / Neutral / Decline to see how outcomes are distributed.
# A team that goes from 0 shot attempts to 1 technically gained momentum,
# but that is very different from a team that goes from 1 to 5.

class_counts = df['Momentum_Class'].value_counts().reindex(['Surge', 'Neutral', 'Decline'])
colors = ['#2ecc71', '#95a5a6', '#e74c3c']

plt.figure(figsize=(8, 5))
bars = plt.bar(class_counts.index, class_counts.values, color=colors, edgecolor='white', linewidth=1.2)
for bar, val in zip(bars, class_counts.values):
    plt.text(bar.get_x() + bar.get_width() / 2, bar.get_height() + 10,
             f'{val}\n({val/len(df):.1%})', ha='center', fontsize=11)
plt.title('Momentum Outcome Breakdown After a Fight', fontsize=14)
plt.xlabel('Outcome (Surge = +2 or more Corsi, Decline = -2 or less)', fontsize=11)
plt.ylabel('Number of Team-Fight Observations', fontsize=11)
plt.grid(True, axis='y', linestyle='--', alpha=0.4)
plt.tight_layout()
plt.savefig('outputs/Momentum_Class_Breakdown.png')
plt.show()

In [ ]:
# --- VISUAL 4: Corsi Before vs After by Game Context ---
# Breaking down by whether the team was trailing and whether they were at home
# gives us a first look at which situations might produce the strongest momentum effect.
# The hypothesis is that trailing teams and home teams benefit most.

fig, axes = plt.subplots(1, 2, figsize=(14, 5))

# By trailing status
trailing_data = df.groupby('Was_Trailing')[['Corsi_Before', 'Corsi_After']].mean()
trailing_data.index = ['Leading / Tied', 'Trailing']
trailing_data.plot(kind='bar', ax=axes[0], color=['#95a5a6', '#2ecc71'],
                   edgecolor='white', rot=0)
axes[0].set_title('Avg Corsi Before vs After by Score State', fontsize=13)
axes[0].set_ylabel('Average Corsi (Shot Attempts)', fontsize=11)
axes[0].set_xlabel('')
axes[0].legend(['Before Fight', 'After Fight'])
axes[0].grid(True, axis='y', linestyle='--', alpha=0.4)

# By home/away
home_data = df.groupby('Is_Home')[['Corsi_Before', 'Corsi_After']].mean()
home_data.index = ['Away', 'Home']
home_data.plot(kind='bar', ax=axes[1], color=['#95a5a6', '#3498db'],
               edgecolor='white', rot=0)
axes[1].set_title('Avg Corsi Before vs After by Venue', fontsize=13)
axes[1].set_ylabel('Average Corsi (Shot Attempts)', fontsize=11)
axes[1].set_xlabel('')
axes[1].legend(['Before Fight', 'After Fight'])
axes[1].grid(True, axis='y', linestyle='--', alpha=0.4)

plt.tight_layout()
plt.savefig('outputs/Corsi_Context_Breakdown.png')
plt.show()

In [ ]:
# --- VISUAL 5: Momentum Effect by Period ---
# Fights early in the game may have a longer window to generate momentum,
# while 3rd period fights happen under more pressure with less time remaining.
# This chart shows whether the average Corsi Delta differs by period.

period_delta = df.groupby('Period_Label')['Corsi_Delta'].mean().reindex(
    ['Period 1', 'Period 2', 'Period 3']
)

colors = ['#2ecc71' if v > 0 else '#e74c3c' for v in period_delta.values]

plt.figure(figsize=(8, 5))
plt.bar(period_delta.index, period_delta.values, color=colors, edgecolor='white', linewidth=1.2)
plt.axhline(0, color='black', linewidth=0.8)
plt.title('Average Corsi Delta by Period', fontsize=14)
plt.xlabel('Period', fontsize=12)
plt.ylabel('Average Corsi Delta (After minus Before)', fontsize=12)
plt.grid(True, axis='y', linestyle='--', alpha=0.4)
plt.tight_layout()
plt.savefig('outputs/Corsi_Delta_By_Period.png')
plt.show()

In [ ]:
# --- VISUAL 6: Has the Momentum Effect Changed Over Time? ---
# As fighting has declined, the remaining fights may be more intentional and situational.
# This checks whether the average xG Delta per fight has shifted across seasons.

season_xg = df.groupby('Season')['xG_Delta'].mean()

plt.figure(figsize=(10, 5))
plt.plot(season_xg.index, season_xg.values, marker='o', linewidth=2.5, color='purple')
plt.axhline(0, color='red', linestyle='--', alpha=0.6, label='No Effect')
plt.title('Average xG Delta Per Fight by Season', fontsize=14)
plt.xlabel('Season', fontsize=12)
plt.ylabel('Mean xG Delta (After minus Before)', fontsize=12)
plt.legend()
plt.grid(True, linestyle='--', alpha=0.4)
plt.tight_layout()
plt.savefig('outputs/xG_Delta_By_Season.png')
plt.show()

---
## Part 2 — Statistical Testing: Paired T-Test

EDA shows us patterns, but it cannot tell us whether those patterns are statistically meaningful or just random variation. For that we use a **paired t-test**.

A paired t-test is the right choice here because every observation has both a before AND an after measurement for the same team in the same game. We are not comparing two separate groups — we are comparing two measurements from the same subject, which makes it a paired design.

We run three separate tests:
- **Corsi** (shot attempt volume)
- **Shots on Goal** (a stricter version of Corsi)
- **Expected Goals** (accounts for shot quality, not just quantity)

The null hypothesis in each case is that there is no difference between before and after. A p-value below 0.05 means we can reject that null hypothesis and conclude the shift is statistically significant.

In [ ]:
# --- PAIRED T-TEST: All Fights ---

metrics = {
    'Corsi': ('Corsi_Before', 'Corsi_After'),
    'Shots on Goal': ('Shots_Before', 'Shots_After'),
    'Expected Goals': ('xG_Before', 'xG_After')
}

print('=' * 55)
print('PAIRED T-TEST RESULTS — ALL FIGHTS')
print('=' * 55)

for metric_name, (before_col, after_col) in metrics.items():
    t_stat, p_val = stats.ttest_rel(df[after_col], df[before_col])
    mean_delta = (df[after_col] - df[before_col]).mean()
    significance = '*** SIGNIFICANT' if p_val < 0.05 else 'not significant'
    print(f'\n{metric_name}')
    print(f'  Mean Delta (After - Before): {mean_delta:.4f}')
    print(f'  T-Statistic: {t_stat:.4f}')
    print(f'  P-Value: {p_val:.4f}  →  {significance}')

print('\n' + '=' * 55)

In [ ]:
# --- SUBGROUP T-TESTS ---
# Even if the overall result is not significant, there may be subgroups where
# fighting does generate a real and meaningful momentum shift.
# We test four subgroups that our EDA suggested might be interesting.

subgroups = {
    'Trailing Teams Only': df[df['Was_Trailing'] == 1],
    'Home Teams Only': df[df['Is_Home'] == 1],
    'Period 1 Fights': df[df['Period'] == 1],
    'Trailing + Home': df[(df['Was_Trailing'] == 1) & (df['Is_Home'] == 1)]
}

print('=' * 60)
print('PAIRED T-TEST RESULTS — SUBGROUPS (Corsi)')
print('=' * 60)

for group_name, group_df in subgroups.items():
    if len(group_df) < 10:
        print(f'\n{group_name}: Too few observations (n={len(group_df)}), skipping')
        continue
    t_stat, p_val = stats.ttest_rel(group_df['Corsi_After'], group_df['Corsi_Before'])
    mean_delta = (group_df['Corsi_After'] - group_df['Corsi_Before']).mean()
    significance = '*** SIGNIFICANT' if p_val < 0.05 else 'not significant'
    print(f'\n{group_name} (n={len(group_df)})')
    print(f'  Mean Corsi Delta: {mean_delta:.4f}')
    print(f'  T-Statistic: {t_stat:.4f}')
    print(f'  P-Value: {p_val:.4f}  →  {significance}')

print('\n' + '=' * 60)

In [ ]:
# --- VISUAL: Before vs After Corsi by Subgroup ---
# Visualizing the same subgroup comparisons makes the results easier to interpret
# and communicate, especially to a non-technical audience.

subgroup_means = pd.DataFrame({
    'Group': list(subgroups.keys()),
    'Before': [g['Corsi_Before'].mean() for g in subgroups.values()],
    'After': [g['Corsi_After'].mean() for g in subgroups.values()]
})

x = range(len(subgroup_means))
width = 0.35

fig, ax = plt.subplots(figsize=(12, 6))
bars1 = ax.bar([i - width/2 for i in x], subgroup_means['Before'],
               width, label='Before Fight', color='#95a5a6', edgecolor='white')
bars2 = ax.bar([i + width/2 for i in x], subgroup_means['After'],
               width, label='After Fight', color='#2ecc71', edgecolor='white')

ax.set_title('Average Corsi Before vs After Fight — by Subgroup', fontsize=14)
ax.set_ylabel('Average Corsi (Shot Attempts)', fontsize=12)
ax.set_xticks(list(x))
ax.set_xticklabels(subgroup_means['Group'], fontsize=10)
ax.legend(fontsize=11)
ax.grid(True, axis='y', linestyle='--', alpha=0.4)
plt.tight_layout()
plt.savefig('outputs/Subgroup_Corsi_Comparison.png')
plt.show()

---
## Part 3 — Predictive Modeling

The t-tests tell us *whether* a momentum shift exists on average. The models tell us *when* it is most likely to happen and what factors drive it.

We build two models:

**Model 1 — Regression (Primary)**
Predicts `xG_Delta` as a continuous value. This is the most informative target because it captures both the direction and magnitude of the momentum shift, and it accounts for shot quality rather than just shot volume.

**Model 2 — Classification (Secondary)**
Predicts `Gained_Momentum` as a binary yes/no. This is simpler and easier to explain, and it serves as a useful companion to the regression model.

Both models use a Random Forest, which is appropriate here because it handles the mix of binary and continuous features well, is robust to small datasets, and produces feature importance scores that are easy to interpret.

In [ ]:
# --- FEATURE PREPARATION ---
# We use the same features for both models.
# Period_Label is encoded as dummy variables so the model can treat each period separately.

features = ['Is_Home', 'Was_Trailing', 'Score_State_At_Fight',
            'Corsi_Before', 'xG_Before', 'Season']

df_model = df.dropna(subset=features + ['xG_Delta', 'Gained_Momentum']).copy()

# Add period dummies
period_dummies = pd.get_dummies(df_model['Period_Label'], prefix='Period', drop_first=True)
df_model = pd.concat([df_model, period_dummies], axis=1)
feature_cols = features + list(period_dummies.columns)

X = df_model[feature_cols]

print(f'Model dataset size: {len(df_model):,} observations')
print(f'Features used: {feature_cols}')

In [ ]:
# --- MODEL 1: REGRESSION — Predicting xG Delta ---
# xG Delta is our primary target because it captures shot quality,
# not just raw shot volume. A team generating high-danger chances after
# a fight is more meaningful than one throwing pucks on net from the perimeter.

y_reg = df_model['xG_Delta']
X_train, X_test, y_train, y_test = train_test_split(X, y_reg, test_size=0.2, random_state=42)

reg_model = RandomForestRegressor(n_estimators=200, max_depth=5,
                                   min_samples_leaf=20, random_state=42)
reg_model.fit(X_train, y_train)
y_pred_reg = reg_model.predict(X_test)

mae = mean_absolute_error(y_test, y_pred_reg)
r2 = r2_score(y_test, y_pred_reg)

print('=' * 45)
print('MODEL 1: REGRESSION — xG Delta')
print('=' * 45)
print(f'Mean Absolute Error: {mae:.4f}')
print(f'R-Squared:           {r2:.4f}')
print()
print('Interpretation:')
print(f'  On average, the model is off by {mae:.4f} xG units when predicting')
print(f'  the momentum shift after a fight.')
print(f'  R-squared of {r2:.2f} means the model explains {r2*100:.1f}% of the variance')
print(f'  in xG Delta — a low R2 here would suggest fighting outcomes are largely random.')
print('=' * 45)

In [ ]:
# --- REGRESSION FEATURE IMPORTANCE ---
# Feature importance tells us which variables the model relies on most
# when predicting how much a team's expected goals will shift after a fight.
# High importance for Was_Trailing would support the hypothesis that
# losing teams benefit most from fights.

reg_importance = pd.DataFrame({
    'Feature': feature_cols,
    'Importance': reg_model.feature_importances_
}).sort_values('Importance', ascending=False)

plt.figure(figsize=(10, 6))
sns.barplot(data=reg_importance, x='Importance', y='Feature', palette='viridis')
plt.title('Feature Importance — Regression Model (xG Delta)', fontsize=14)
plt.xlabel('Importance Score', fontsize=12)
plt.ylabel('Feature', fontsize=12)
plt.grid(True, axis='x', linestyle='--', alpha=0.5)
plt.tight_layout()
plt.savefig('outputs/Feature_Importance_Regression.png')
plt.show()

In [ ]:
# --- MODEL 2: CLASSIFICATION — Predicting Gained Momentum ---
# The binary target is simpler and easier to communicate.
# We optimize the decision threshold using F1-score, the same approach
# used in the goalie pull project, because the classes are imbalanced
# (more teams gain momentum than not in some subgroups).

y_clf = df_model['Gained_Momentum']
X_train_c, X_test_c, y_train_c, y_test_c = train_test_split(X, y_clf, test_size=0.2, random_state=42)

clf_model = RandomForestClassifier(n_estimators=200, max_depth=5,
                                    min_samples_leaf=20, random_state=42)
clf_model.fit(X_train_c, y_train_c)

# Optimize threshold via F1
y_prob = clf_model.predict_proba(X_test_c)[:, 1]
precision, recall, thresholds = precision_recall_curve(y_test_c, y_prob)
f1_scores = np.divide(
    2 * (precision[:-1] * recall[:-1]),
    (precision[:-1] + recall[:-1]),
    out=np.zeros_like(thresholds),
    where=(precision[:-1] + recall[:-1]) != 0
)
best_threshold = thresholds[np.argmax(f1_scores)]
y_pred_clf = (y_prob >= best_threshold).astype(int)

tn, fp, fn, tp = confusion_matrix(y_test_c, y_pred_clf).ravel()
accuracy = (tp + tn) / (tp + tn + fp + fn)
sensitivity = tp / (tp + fn) if (tp + fn) > 0 else 0
specificity = tn / (tn + fp) if (tn + fp) > 0 else 0

print('=' * 50)
print('MODEL 2: CLASSIFICATION — Gained Momentum')
print('=' * 50)
print(f'Optimal F1 Threshold: {best_threshold:.4f}')
print(f'Accuracy:             {accuracy:.4f}')
print(f'Sensitivity:          {sensitivity:.4f}')
print(f'Specificity:          {specificity:.4f}')
print(f'Confusion Matrix: TN={tn}, FP={fp}, FN={fn}, TP={tp}')
print('=' * 50)

In [ ]:
# --- CLASSIFICATION FEATURE IMPORTANCE ---

clf_importance = pd.DataFrame({
    'Feature': feature_cols,
    'Importance': clf_model.feature_importances_
}).sort_values('Importance', ascending=False)

plt.figure(figsize=(10, 6))
sns.barplot(data=clf_importance, x='Importance', y='Feature', palette='magma')
plt.title('Feature Importance — Classification Model (Gained Momentum)', fontsize=14)
plt.xlabel('Importance Score', fontsize=12)
plt.ylabel('Feature', fontsize=12)
plt.grid(True, axis='x', linestyle='--', alpha=0.5)
plt.tight_layout()
plt.savefig('outputs/Feature_Importance_Classification.png')
plt.show()

---
## Summary of Findings

This section is intentionally left for you to fill in after running the notebook with the full dataset. The key things to document are:

**From the T-Tests:**
- Was the overall Corsi shift statistically significant?
- Which subgroup (if any) showed the strongest and most significant effect?
- Did xG tell a different story than raw Corsi?

**From the Models:**
- What was the R-squared of the regression model? A very low R2 (close to 0) is itself an interesting finding — it suggests fight outcomes are largely unpredictable from game context alone.
- Which features were most important in both models?
- Did trailing teams or home teams show up as meaningful predictors?

**The Bigger Picture:**
- Does the data support the conventional wisdom that fighting generates momentum?
- Are there specific situations where it clearly does or does not?
- How has the effect (if any) changed as fighting has declined since 2010?